# 시계열 분석

지금까지 다룬 데이터: 대부분 횡단면 (한 시점에 여러 대상)
- 이제 질문: **"시간 순서 자체가 정보인 데이터는 어떻게 다루지?"**

- 흐름
  - 시계열 이해: 어떻게 생겼고 뭐가 특별한지
  - 전처리: 결측치, 이상치, 평활화, 요소분해
  - 정상성 판단: 모델에 넣을 수 있는 상태인지 확인 & 만들기
  - 모델링 & 예측: 단변량 / 다변량


# 시계열 데이터란?

### 정의
- **시간 흐름에 따라 일정한 간격, 순서대로 관측·기록된 데이터**
- 주가, 기온, 매출처럼 같은 대상을 시간에 따라 반복 기록하면 다 시계열
- 표기: Y1, Y2, …, Yt, …, YT
  - t: 시점 / Yt: t 시점 값 / T: 마지막 시점
- 조건
  - 시간 순서가 의미 있음 → 섞으면 의미 사라짐
  - 관측 간격이 일정함 (이 간격 = 주기)
- vs 횡단면 데이터
  - 횡단면: 한 시점에 여러 대상 (서강대생 100명 키) → 순서 섞어도 상관 x
  - 시계열: 한 대상을 여러 시점 (식당 A 월별 매출) → 섞으면 "언제 올랐는지" 정보 망가짐
- 단변량 vs 다변량
  - 단변량: 변수 하나, 자기 과거만 보고 예측 (삼성전자 일별 종가)
  - 다변량: 여러 변수 같이 봄 (아메리카노 판매량 ← 기온, 강수량, 요일)
  - 다변량이 무조건 좋은 건 x → 내일 판매량 예측하려면 내일 기온도 예측해야 함
  - 그래서 요일, 공휴일처럼 미리 아는 변수가 자주 쓰임
- 목적: 미래 예측 (Forecasting)
  - 기본 가정: 과거 패턴이 미래에도 어느 정도 반복됨
  - 그래서 코로나처럼 처음 보는 일은 예측 어려움, 먼 미래일수록 불확실성 ↑
  - 예측 자체보다 예측으로 "미리" 결정하는 게 목적 (재고, 발전량, 서버 증설)

### 특징
- 시간의존성: 과거 값이 현재 값에 영향, 가까운 과거일수록 영향 큼
  - 데이터 누수 주의: 랜덤으로 train/test 나누면 5월로 학습해서 3월 맞히는 꼴 → 성능 뻥튀기
  - → 반드시 과거로 학습, 미래로 평가
- 계절성 & 추세
  - 계절성: 일정 주기로 비슷한 패턴 반복 → "언제마다?"
  - 추세: 장기적으로 증가/감소하는 방향 → "어느 방향?"
  - 보통 둘 다 같이 나옴 (항공기 승객: 증가 추세 + 여름마다 최고)
- 자기상관성: **자기 자신의 과거와 상관관계를 갖는 특성** ← 제일 중요
  - 오늘 판매량 vs 어제 판매량의 상관계수
  - -1 ~ 1. 1에 가까우면 과거에 높았으면 지금도 높음
  - 시차(lag): 몇 시점 전과 비교하는지 (일별 데이터면 어제 = 시차1, 일주일 전 = 시차7)
  - 시차 바꿔가며 계산하면 반복 주기 찾기 ㄱㄴ (12개월 전과 매번 비슷 → 1년 주기)
  - 한 줄: 과거 값으로 지금 값을 짐작할 수 있는 정도
- 자기상관 있으면 일반 회귀 그대로 쓰면 x
  - 오차항 독립성 가정 위반 → p-value 왜곡 (안 유의한 변수가 유의해 보임)
  - 허위 상관: 둘 다 시간 따라 오르면 관계 없어도 상관 ↑ (넷플릭스 가입자 vs 서울 집값)
  - 예측 오차 누적, 과거 패턴이 계속 유지된다는 보장 x
  - → 시계열 전용 모델 쓰거나, 추세·계절성 걷어낸 뒤 분석

### 분석요인
- 시계열 = 추세 + 계절 + 순환 + 불규칙이 섞인 것
- 추세(T): 장기적인 증가/감소 (1인 가구 수 증가, 출생아 수 감소)
- 계절(S): 1년 이내 고정 주기로 반복. 날씨, 달력, 관습 때문 (여름 에어컨, 월급날 카드값)
- 순환(C): 1년보다 길고 주기가 일정 x (경기 호황/불황, 반도체 업황)
  - 계절 vs 순환: "매년 8월에 오른다"처럼 달력으로 말할 수 있으면 계절, 언제 올지 모르면 순환
  - 찾기 어렵고 추세랑 구분도 어려움 → 보통 추세에 포함 (추세-순환)
- 불규칙(I): 나머지. 잔차, 노이즈 (폭우로 매출 급감, 유명인 방문으로 손님 급증)
  - 이상적으로는 랜덤하게 흩어져야 함, 유독 튀면 이상치일 수도


# 시계열 데이터 전처리

- 현실 데이터: 빠진 값, 튀는 값, 노이즈 → 그대로 넣으면 잘못된 패턴 학습
- 시계열은 값 하나가 앞뒤랑 이어져 있어서 잘못된 값 하나가 주변 분석까지 망침
- 순서: 데이터 확인 → 결측치 → 이상치 → 평활화 → 요소분해 → 정상성

### 결측치
- 행 지우면 관측 간격 깨짐, 전체 평균으로 채우면 시간 흐름 무시
- → 앞뒤 흐름 고려해서 채워야 함
- 방법: 삭제 / 단순 대치(평균, 중앙값) / 보간 / 이동평균 대치 / 머신러닝(KNN, 회귀)
- 보간법: 앞뒤 관측값으로 사이 값 추정 ← 시계열에서 제일 많이 씀
  - 2일 110, 4일 130이면 직선으로 이어서 3일 = 120
  - 선형: 직선으로 연결. 빠르고 간단, 급변하는 데이터엔 부정확
  - 다항식: 다항식으로 근사. 비선형에 좋지만 고차면 과적합
  - 스플라인: 구간별 다항식(보통 3차, cubic spline). 부드럽고 안정적
  - 조각별 상수: ffill(직전 값), bfill(직후 값)
  - 최근접 이웃: 거리상 제일 가까운 값으로
- 회귀 vs 보간
  - 회귀: 점들 사이를 제일 잘 지나는 선 → 점을 안 지날 수도 (오차 허용)
  - 보간: 관측값을 정답으로 믿고 반드시 지남
  - 있는 값 바꾸면 안 되니까 결측치엔 보간
- 고르는 법: 완만 → 선형 / 곡선 → 스플라인 / 계단 모양(가격표) → ffill / 모르겠으면 선형부터

### 이상치
- 언제 나왔는지가 중요
  - 35도: 8월이면 정상, 1월이면 이상치
  - → 추세, 계절성 고려해서 판단
- 통계 기반
  - IQR: Q1−1.5×IQR 미만, Q3+1.5×IQR 초과면 이상치. 정규분포 아니어도 ㄱㄴ, 추세·계절 강하면 부적합
  - Z-score: 절댓값 3 넘으면 이상치. 간단하지만 정규분포 가정
    - 큰 이상치가 평균, 표준편차를 끌어올려서 오히려 안 잡힐 수도 → Rolling Z-score
  - STL 분해: 추세, 계절 빼고 남은 잔차가 기준 넘으면 이상치. 계절성 뚜렷할 때 좋음
- 모델 기반
  - CART: 다음 값 예측해서 오차 크면 이상치
  - 클러스터링: 어느 군집에도 안 속하면 이상치 (K-means, DBSCAN). 라벨 x
  - 오토인코더: 정상만 학습 → 이상치는 복원 못 해서 복원 오차 ↑
    - LSTM 오토인코더면 시간 의존성까지 학습
- 고르는 법: IQR, Z-score부터 → 계절성 뚜렷하면 STL → 크고 복잡하면 모델 기반
- 처리: 오류인지 실제 사건인지 먼저 확인
  - 바로 지우면 간격 깨짐 → NaN으로 바꾸고 보간해서 채우는 경우 많음

### 평활화
- 들쭉날쭉한 노이즈 줄여서 큰 흐름 보기 → 주로 EDA에서 씀
- 이동평균: N개씩 묶어서 평균, 한 칸씩 옮기며 계산
  - N=3이면 100, 130, 110, 140, 120 → 113.3, 126.7, 123.3
  - N 작음: 원래 데이터랑 비슷, 빠르게 반응, 노이즈 남음
  - N 큼: 매끄럽고 큰 흐름 잘 보임, 변화에 늦게 반응
  - N을 계절 주기랑 맞추면 계절성 상쇄 (월별 데이터에 12개월 이동평균 → 추세만 남음)
- 종류
  - SMA: 다 같은 가중치. 간단하지만 최근 변화에 늦음
  - WMA: 최근 값일수록 가중치 ↑. 가중치를 직접 정해야 함
  - EMA: 과거로 갈수록 가중치가 지수적으로 ↓
    - EMAt = αYt + (1−α)EMAt−1, α 클수록 최근 값 중시

### 요소분해
- 섞여 있는 시계열을 추세, 계절, 잔차로 다시 나누는 것 (순환은 추세에 포함)
- 아이스크림 가게: 입소문으로 매년 우상향(T) + 여름에 잘 팔림(S) + 먹방 유튜버 방문(R)
- 장점: 패턴 이해, 변동 원인 분석, 더 정교한 분석

- 가법 vs 승법
  - 가법: Yt = Tt + St + Rt. 계절 효과 크기가 일정 (매년 여름 +50잔)
  - 승법: Yt = Tt × St × Rt. 추세 커지면 계절 효과도 비례해서 커짐 (매년 여름 ×1.2)
  - 그래프 물결 높이 일정 → 가법 / 점점 커짐 → 승법
  - 현실의 매출, 주가, 방문자 수는 승법에 가까운 경우 많음
  - 승법에 로그 취하면 log Yt = log Tt + log St + log Rt → 가법처럼 다룰 수 있음

- seasonal_decompose (MA 기반)
  1. 중심 이동평균으로 추세(T)
  2. 추세 제거 (가법 Y−T / 승법 Y÷T)
  3. 같은 계절 위치끼리 평균 → 계절(S)
  4. 남은 게 잔차(R)
  - 제일 기본적이고 전통적인 방법
- STL (LOESS 기반)
  - LOESS: 지점마다 주변 데이터로 작은 회귀선 → 이어 붙여서 부드러운 곡선 → 시간 따라 바뀌는 추세, 계절도 잡음
  - Inner Loop: 추세, 계절을 번갈아 추정하면서 다듬음
  - Outer Loop: 잔차 큰 시점에 작은 가중치 → 이상치 영향 ↓ (강건한 분해)
- 한 줄: 결합 방식 → 가법/승법, 계산 방법 → seasonal_decompose(단순) / STL(유연, 강건)


# 정상성 판단

### 정상성이란
- **통계적 특성(평균, 분산, 자기상관)이 시간에 따라 변하지 않는 성질**
- 어느 시점에서 잘라 봐도 비슷한 모습
- 정상: 주가 등락률, 기계 정상 가동 중 진동값
- 비정상: 주가 가격 자체, 연도별 GDP, 월별 항공기 승객 수
- 왜 중요? 시계열 예측의 가정(과거 패턴 반복)이 성립하려면 규칙이 안 변해야 함
  - 기출 비유: 매년 범위, 난이도가 비슷해야 기출 푼 게 의미 있음
- 현실 데이터는 대부분 비정상 → 판단하고 확보하는 과정 필요

### 강한 정상성 vs 약한 정상성
- 강한: 확률분포 자체가 시간 따라 전혀 안 변함 (왜도, 첨도까지 전부) → 현실엔 거의 x
- 약한: 아래 3개만 만족하면 됨
  1. 평균 일정
  2. 분산 일정 (무한대 x)
  3. 자기공분산이 시차(h)에만 의존 (1월 1일~2일 관계 = 7월 1일~2일 관계)
- AR, MA, ARIMA 등 대부분 모델이 가정하는 건 약한 정상성

### 판단 - 시각
- 정상: 평균 중심으로 오르내림, 폭 비슷, 벗어나도 돌아옴(평균 회귀), 주기 반복 x
- 비정상
  - 추세: 계속 오르거나 내림 → 평균 변함
  - 계절성: 주기마다 같은 모양 반복 → 시점마다 평균 다름
  - 분산 변화: 폭이 점점 커지거나 작아짐 (나팔 모양)
- ACF로도 확인: 정상이면 막대가 몇 칸 만에 0 근처로, 비정상이면 아주 천천히 줄어듦
- 근데 보는 사람마다 판단 다름 → 통계 검정으로 객관적으로

### 판단 - 통계
- 단위근: 시계열을 나타내는 식의 근이 1 → 과거 충격이 영구적으로 누적
  - Yt = φYt−1 + εt (φ = 어제 값이 오늘에 얼마나 남는지)
  - φ = 0.5: 충격 10 → 5 → 2.5 … 사라짐 → 평균으로 돌아옴 → 정상
  - φ = 1: 충격 그대로 남고 새 충격이 계속 쌓임 → 안 돌아오고 분산 ↑ → 비정상
- ADF: 귀무가설 = 단위근 있음(비정상). p < 0.05면 기각 → 정상
- KPSS: 귀무가설 = 정상. p < 0.05면 기각 → 비정상
- 재판 비유: 출발점(귀무가설)이 반대라서 같이 쓰면 서로 보완
- 결과 조합
  - 둘 다 정상 → 정상 시계열, 바로 모델링
  - 둘 다 비정상 → 차분 등으로 정상성 확보 후 재검정
  - ADF 비정상 + KPSS 정상 → 추세 정상 → 추세 제거 후 재검정
  - ADF 정상 + KPSS 비정상 → 차분 정상 → 차분 후 재검정

### 정상성 확보 방법
- 깨지는 원인 3개: 추세, 계절성, 분산 변화 → 하나씩 제거
- 추세 → 차분: ΔYt = Yt − Yt−1
  - 100, 112, 119, 133, 140 → +12, +7, +14, +7 → 변화량은 10 근처라 평균 일정
  - 단위근 시계열 차분하면 "오늘 새로 생긴 일"만 남음
  - 2차 차분: 오르는 속도 자체가 빨라질 때. 많이 하면 없던 패턴 생김 → 보통 2차까지, 최소 횟수만
  - 역차분: 예측 결과가 변화량이라 원래 단위로 되돌려야 함
- 계절성 → 계절 차분: Yt − Yt−m (올해 8월 − 작년 8월 → 계절 효과 상쇄)
- 분산 변화 → 로그 변환
  - 큰 값은 많이, 작은 값은 조금 줄임 (10, 100, 1000 → 1, 2, 3)
  - 0, 음수엔 x → 1 더하고 로그
  - Box-Cox 변환도 있음
- 결국 정상성 확보 → 패턴 파악 → 모델링의 순환


# 단변량 시계열 분석

- 변수 하나의 기록으로 자기 과거 보고 예측
- 값(y) vs 충격(ε)
  - 값: 장부에 적힌 최종 결과, 과거가 섞여 있음
  - 충격: 그날 새로 생긴 예측 불가능한 변동, 안 보임 (방송 소개, 폭우, 이유 없이 잘 팔린 날)
  - 주말 효과, 미리 공지한 할인 행사처럼 미리 아는 건 충격 x

## 안정적 시계열 분석
- 정상 시계열 = 돌아갈 "평소"가 있는 시계열 (오래된 동네 가게)

### AR
- 오늘 값을 **자기 과거 값**으로 설명 → 자기상관 활용
- AR(p): yt = c + φ1yt−1 + … + φpyt−p + εt
- 손님이 어제 손님 수 보고 오는 가게 (입소문)
- 방송 충격 100, 계수 0.8 → 100, 80, 64, 51 … 끝없이 이어지며 옅어짐
  - 어제 값 안에 그저께 값이 들어 있어서

### MA
- 오늘 값을 **과거 충격**으로 설명
- MA(q): yt = μ + εt + θ1εt−1 + … + θqεt−q
- 손님이 어제 방송 보고 오는 가게
- 충격 100, 계수 0.8 → 100, 80, 0, 0 → q일 뒤 딱 끊김
- 오늘 충격은 아직 모르니 예측할 땐 0으로 둠

### AR, MA 전제조건
- 정상성: 충격 받아도 평소로 돌아오나?
  - AR: 계수 조건 필요. AR(1)이면 |φ| < 1 (1.1이면 110, 121로 불어남)
    - AR(p)면 특성방정식 1 − φ1z − … − φpz^p = 0의 모든 근 절댓값 > 1
  - MA: 항상 만족 (q일 뒤 영향 끝나니까)
  - 단위근 검정(ADF, KPSS)이랑 다름 → 그건 데이터를 보는 거, 이건 추정한 계수를 보는 거
- 가역성: 판매량 기록으로 지난 충격을 거꾸로 계산 ㄱㄴ?
  - AR: 항상 만족
  - MA: 계수 조건 필요. MA(1)이면 |θ| < 1
    - 첫날 충격을 0으로 어림하고 시작 → 계수 0.5면 오차 2, 1, 0.5로 사라짐, 2면 8, 16, 32로 폭발

### ARMA
- AR + MA. 과거 값 + 과거 충격 둘 다 영향
- AR 0.5, MA 0.4, 충격 100 → 100, 90, 45, 22.5
  - 다음 날 90 = 입소문 50 + 방송 늦게 본 40
- AR만으로 설명하려면 시차 많이 필요 → ARMA는 더 적은 계수로 설명
- AR 부분은 정상성, MA 부분은 가역성 만족해야 함
- ARMA(p,0) = AR(p) / ARMA(0,q) = MA(q) / ARMA(0,0) = 백색잡음

### 차수 결정: ACF, PACF
- ACF: 오늘 값과 k시점 전 값의 상관. 중간 시점 거쳐 온 영향까지 포함 → q 결정
- PACF: 중간 시점 영향 빼고 직접 상관 → p 결정
- 점선(보통 ±1.96/√n) 안쪽이면 0으로 봄
- 고르는 법: 막대가 급격히 줄어들기 직전의 시차
  - ACF 천천히 감소, PACF p+1부터 0 → AR(p)
  - ACF q+1부터 0, PACF 천천히 감소 → MA(q)
  - 둘 다 끊김 → ARMA(p, q)

## 불안정 시계열 분석
- 돌아갈 평소가 없음 (새로 생겨서 판매량이 계속 느는 가게)
- 추세 → 차분(d) → ARIMA / 계절성 → 계절 차분(D) → SARIMA
- 키 비유: 키 기록 그대로 넣으면 "내년에 줄어든다" 예측 → 1년간 자란 길이(차분)엔 평소 있음
- 모형을 고치는 게 아니라 데이터를 모형이 다룰 수 있게 바꿔주는 것

### ARIMA(p, d, q)
- ARMA + 차분
  1. 차분해서 증감 만들기
  2. 증감을 ARMA로 예측
  3. 예측한 증감을 마지막 판매량에 더해서 되돌리기
- 100, 103, 104, 108, 110 → 증감 +3, +1, +4, +2 → 다음 증감 +2.5 예측 → 110 + 2.5 = 112.5
- 특수한 경우
  - ARIMA(0,0,0) = 백색잡음
  - ARIMA(p,0,0) = AR(p) / ARIMA(0,0,q) = MA(q)
  - 랜덤워크: 상수 없는 ARIMA(0,1,0) → Yt = Yt−1 + εt → 내일 최선의 예측 = 오늘 값
  - 표류 포함 랜덤워크: 상수 있는 ARIMA(0,1,0) → Yt = Yt−1 + c + εt → 떠돌면서 한 방향으로

### SARIMA(p, d, q)(P, D, Q)m
- ARIMA + 계절 차분, 계절 AR, 계절 MA
- "지난달 대비"에 "작년 같은 달 대비" 추가
- 붕어빵: 10월, 11월 둘 다 작년보다 50씩 더 팔림 → 12월도 작년 900 + 50 = 950
- p, d, q: ARIMA 그대로 / P, D, Q: 한 주기 전 기준으로 똑같이 / m: 한 주기가 몇 칸 (월별 12, 분기별 4, 요일 7)
- 계절성 있으면 ACF가 12, 24, 36에서 다시 솟음 → 계절 차분 먼저!

### 하이퍼파라미터 정하는 법
- ACF, PACF는 실제로 깔끔하게 안 나오는 경우 많음
- d, D: ADF로 정상 나올 때까지 차분, 그 횟수. 대부분 0 아니면 1
- p, q: ACF, PACF로 범위 잡고 → AIC, BIC로 그리드 서치 (낮을수록 좋음, 보통 0~3)


# 다변량 시계열 분석

- 변수 여러 개를 자기 과거 + 다른 변수 과거로 같이 예측
  - 어제 판매량, 어제 광고비 → 오늘 판매량, 오늘 광고비

### VAR
- AR을 변수 여러 개로 넓힌 것
- VAR(1): 변수마다 식 하나, 각 식에 모든 변수의 어제 값
  - 오늘 판매량 = 어제 판매량 × 0.6 + 어제 광고비 × 0.3 + 충격
  - 오늘 광고비 = 어제 광고비 × 0.5 − 어제 판매량 × 0.2 + 충격
  - 0.3: 광고가 판매 끌어올림 / −0.2: 잘 팔리면 광고 줄임
- 적합: 서로 영향 주고받음(양방향), 변수 적고 데이터 충분, 다 정상이거나 차분하면 정상
- 부적합
  - 한 방향 영향 (기온 → 판매량, 반대는 x) → 단변량 모형에 기온을 설명변수로
  - 변수 많은데 데이터 적음 (변수 2개면 계수 4개, 3개면 9개)
- 변수 고르기: Granger 인과관계 검정
  - "X의 과거가 Y 예측에 도움 되나?"
  - Y 과거만 vs Y 과거 + X 과거 비교, p < 0.05면 포함
  - Granger 인과 ≠ 진짜 인과 (닭 울음 → 해 뜸, 시간 선후관계만)
  - 양방향으로 검정
- 분석 도구
  - 충격 반응 함수(IRF): 한 변수에 충격 주면 다른 변수가 시간 따라 어떻게 반응?
    - 광고 충격 100 → 판매량 0, 30, 33, 25.5 … → 광고 효과는 이틀 뒤 최대
  - 예측오차 분산 분해: 예측 빗나간 게 어느 쪽 충격 탓인지 비율로
    - 판매량 예측: 1일 뒤 100% 판매 쪽 → 5일 뒤 판매 83%, 광고 17%
  - IRF = "어떻게 반응하나" / 분산 분해 = "누구 영향이 얼마나 크나"


# 정리

- 흐름: 시계열 이해 → 전처리(결측치, 이상치, 평활화, 요소분해) → 정상성 판단(시각, ADF/KPSS) → 확보(차분, 계절 차분, 로그) → 모델링
- 상황별 모형
  - 변수 1개 + 정상 → AR, MA, ARMA
  - 변수 1개 + 추세 → ARIMA
  - 변수 1개 + 계절성 → SARIMA
  - 변수 여러 개 + 서로 영향 → VAR
